# RQ4: Does Inherent Model Interpretability Improve NLE Quality?

**Research Question**: Does SARIMAX's inherent interpretability (coefficients) lead to better natural language explanations compared to black-box ML models?

**Background**:
- SARIMAX is a classical time series model with directly interpretable coefficients
- ML models (XGBoost, RF, MLP) are black-boxes requiring XAI methods (SHAP, LIME)
- SARIMAX only supports XAI="none" in our design (N=66)
- Conventional wisdom: simpler/interpretable models should yield better explanations

**Critical Observation**:
- Model R² values: XGBoost (0.69) > SARIMAX (0.55) > RF (0.37) > MLP (0.21)
- If R² drove NLE quality, we'd expect: XGBoost > SARIMAX > RF > MLP
- But actual NLE ranking is: XGBoost > MLP > RF > SARIMAX
- **SARIMAX underperforms models with LOWER R²!**

**Key Analyses**:
1. Fair comparison: SARIMAX vs ML models at XAI="none" only
2. Per-dimension analysis: SARIMAX vs each ML model
3. Testing the R² hypothesis: Does prediction quality drive NLE quality?
4. The model TYPE hypothesis: Classical vs ML structure
5. LLM and Strategy effects within SARIMAX

---

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Statistical packages
from scipy import stats
from scipy.stats import f_oneway, ttest_ind, mannwhitneyu, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from statsmodels.formula.api import ols
import pingouin as pg

# Paths
EVAL_DIR = Path("../../Data/forsetzung_results/20250813_135743")

print(f"Evaluation dir exists: {EVAL_DIR.exists()}")

Evaluation dir exists: True


In [2]:
# Load G-Eval evaluation data from BOTH judges
geval_gpt4_raw = pd.read_csv(EVAL_DIR / "geval_gpt4.csv")
geval_deepseek_raw = pd.read_csv(EVAL_DIR / "geval_deepseek.csv")

# Dimension mappings - 4 dimensions (Prediction Closeness excluded)
DIMS = ['accuracy', 'lay_user_relevancy', 'expert_relevancy', 
        'usefulness_explanation_helpfulness']

DIM_NAMES = {
    'accuracy': 'Accuracy',
    'lay_user_relevancy': 'Lay Relevancy',
    'expert_relevancy': 'Expert Relevancy',
    'usefulness_explanation_helpfulness': 'Helpfulness'
}

# Create unified score columns
geval_gpt4 = geval_gpt4_raw.copy()
geval_deepseek = geval_deepseek_raw.copy()

SCORE_COLS = []
for dim in DIMS:
    unified_col = f'score_{dim}'
    SCORE_COLS.append(unified_col)
    geval_gpt4[unified_col] = geval_gpt4_raw[f'eval_{dim}_g_eval_score']
    geval_deepseek[unified_col] = geval_deepseek_raw[f'eval_{dim}_traditional_score']

FACTOR_COLS = ['LLM', 'Model', 'XAI', 'Strategy']

# Merge both judges by averaging
df_eval = geval_gpt4[FACTOR_COLS].copy()
for col in SCORE_COLS:
    df_eval[col] = (geval_gpt4[col] + geval_deepseek[col]) / 2

print(f"Total dataset: N={len(df_eval)}")
print(f"\nModel distribution:")
print(df_eval.groupby('Model').size())
print(f"\nXAI distribution:")
print(df_eval.groupby('XAI').size())
print(f"\nModel × XAI cross-tabulation:")
display(pd.crosstab(df_eval['Model'], df_eval['XAI']))

Total dataset: N=660

Model distribution:
Model
MLP             198
RandomForest    198
SARIMAX          66
XGB             198
dtype: int64

XAI distribution:
XAI
lime    198
none    264
shap    198
dtype: int64

Model × XAI cross-tabulation:


XAI,lime,none,shap
Model,,,
MLP,66,66,66
RandomForest,66,66,66
SARIMAX,0,66,0
XGB,66,66,66


In [3]:
# Statistical Helper Functions

def cohens_d(g1, g2):
    """Cohen's d effect size for two groups."""
    n1, n2 = len(g1), len(g2)
    var1, var2 = g1.var(), g2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (g1.mean() - g2.mean()) / pooled_std if pooled_std > 0 else 0

def omega_sq_from_anova(*groups):
    """Compute omega-squared from group data."""
    k = len(groups)
    n_total = sum(len(g) for g in groups)
    df_between = k - 1
    df_within = n_total - k
    grand_mean = np.concatenate(groups).mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_within = sum(((g - g.mean())**2).sum() for g in groups)
    ms_within = ss_within / df_within
    omega_sq = (ss_between - df_between * ms_within) / (ss_between + ss_within + ms_within)
    return max(0, omega_sq)

def effect_label(d):
    d = abs(d)
    if d < 0.2: return "negl."
    elif d < 0.5: return "small"
    elif d < 0.8: return "medium"
    else: return "large"

def omega_label(w2):
    if w2 < 0.01: return "negl."
    elif w2 < 0.06: return "small"
    elif w2 < 0.14: return "medium"
    else: return "large"

def fdr_correct(p_values, alpha=0.05):
    reject, p_corrected, _, _ = multipletests(p_values, alpha=alpha, method='fdr_bh')
    return p_corrected, reject

# Correct R² values from Table 1 in paper
MODEL_R2 = {'XGB': 0.69, 'RandomForest': 0.37, 'MLP': 0.21, 'SARIMAX': 0.55}

print("Helper functions loaded.")

Helper functions loaded.


---

## 1. Descriptive Statistics: SARIMAX Overview

First, let's understand the SARIMAX data and compare to ML models.

In [4]:
# Separate SARIMAX and ML data
df_sarimax = df_eval[df_eval['Model'] == 'SARIMAX'].copy()
df_ml = df_eval[df_eval['Model'].isin(['XGB', 'RandomForest', 'MLP'])].copy()

print("="*80)
print("SARIMAX Overview")
print("="*80)
print(f"\nSARIMAX: N={len(df_sarimax)}")
print(f"ML models: N={len(df_ml)}")

# SARIMAX only has XAI='none'
print(f"\nSARIMAX XAI distribution: {df_sarimax['XAI'].value_counts().to_dict()}")
print(f"SARIMAX LLM distribution: {df_sarimax['LLM'].value_counts().to_dict()}")
print(f"SARIMAX Strategy distribution: {df_sarimax['Strategy'].value_counts().to_dict()}")

# Descriptive stats
print("\n" + "="*80)
print("Descriptive Statistics by Model")
print("="*80)

for model in ['XGB', 'RandomForest', 'MLP', 'SARIMAX']:
    subset = df_eval[df_eval['Model'] == model]
    r2 = MODEL_R2[model]
    print(f"\n{model} (R²={r2}, N={len(subset)}):")
    for col in SCORE_COLS:
        dim = DIM_NAMES[col.replace('score_', '')]
        mean = subset[col].mean()
        std = subset[col].std()
        print(f"  {dim}: {mean:.2f} ± {std:.2f}")

SARIMAX Overview

SARIMAX: N=66
ML models: N=594

SARIMAX XAI distribution: {'none': 66}
SARIMAX LLM distribution: {'GPT': 24, 'L3_LOCAL': 24, 'DEEPSEEK': 18}
SARIMAX Strategy distribution: {'meta_prompting': 9, 'few_shot': 9, 'role_based': 9, 'reflexion': 9, 'self_consistency': 9, 'zero_shot': 9, 'cot_few_shot': 6, 'cot_zero_shot': 6}

Descriptive Statistics by Model

XGB (R²=0.69, N=198):
  Accuracy: 4.56 ± 0.55
  Lay Relevancy: 4.53 ± 0.48
  Expert Relevancy: 4.00 ± 0.58
  Helpfulness: 4.43 ± 0.48

RandomForest (R²=0.37, N=198):
  Accuracy: 4.53 ± 0.55
  Lay Relevancy: 4.29 ± 0.55
  Expert Relevancy: 3.82 ± 0.53
  Helpfulness: 4.27 ± 0.52

MLP (R²=0.21, N=198):
  Accuracy: 4.50 ± 0.61
  Lay Relevancy: 4.12 ± 0.52
  Expert Relevancy: 3.86 ± 0.61
  Helpfulness: 4.29 ± 0.50

SARIMAX (R²=0.55, N=66):
  Accuracy: 4.47 ± 0.66
  Lay Relevancy: 4.04 ± 0.52
  Expert Relevancy: 3.76 ± 0.69
  Helpfulness: 4.12 ± 0.44


---

## 2. Fair Comparison: SARIMAX vs ML at XAI="none"

**Critical Analysis**: Since SARIMAX only has XAI="none", we must compare it to ML models at XAI="none" to isolate the model interpretability effect from XAI effects.

This answers: **Does inherent interpretability help when no XAI is provided?**

In [5]:
# Filter to XAI='none' only for fair comparison
df_none = df_eval[df_eval['XAI'] == 'none'].copy()

print("="*80)
print("FAIR COMPARISON: All models at XAI='none'")
print("="*80)

print(f"\nTotal at XAI='none': N={len(df_none)}")
print(f"\nModel distribution at XAI='none':")
print(df_none.groupby('Model').size())

# Descriptive stats at XAI='none'
print("\n" + "-"*60)
print("Mean scores by Model (XAI='none' only)")
print("-"*60)

means_none = df_none.groupby('Model')[SCORE_COLS].mean()
means_none.columns = [DIM_NAMES[c.replace('score_', '')] for c in means_none.columns]
means_none['Overall'] = means_none.mean(axis=1)
means_none['R²'] = means_none.index.map(MODEL_R2)
means_none = means_none.sort_values('R²', ascending=False)
display(means_none.round(3))

FAIR COMPARISON: All models at XAI='none'

Total at XAI='none': N=264

Model distribution at XAI='none':
Model
MLP             66
RandomForest    66
SARIMAX         66
XGB             66
dtype: int64

------------------------------------------------------------
Mean scores by Model (XAI='none' only)
------------------------------------------------------------


,Accuracy,Lay Relevancy,Expert Relevancy,Helpfulness,Overall,R²
Model,,,,,,
XGB,4.509,4.605,3.890,4.539,4.386,0.69
SARIMAX,4.465,4.037,3.756,4.121,4.094,0.55
RandomForest,4.457,4.263,3.683,4.294,4.174,0.37
MLP,4.501,4.072,3.763,4.315,4.163,0.21


In [6]:
# One-way ANOVA at XAI='none'
print("="*80)
print("ANOVA: Model effect at XAI='none' (Fair Comparison)")
print("="*80)

models = ['XGB', 'RandomForest', 'MLP', 'SARIMAX']
model_groups_none = {m: df_none[df_none['Model'] == m] for m in models}

print(f"\n{'Dimension':<18} {'F':>8} {'p':>10} {'ω²':>8} {'Effect':>8} {'Sig'}")
print("-"*60)

anova_results = []
for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    groups = [model_groups_none[m][col].values for m in models]
    f_stat, p_val = f_oneway(*groups)
    w2 = omega_sq_from_anova(*groups)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
    anova_results.append({'dim': dim, 'F': f_stat, 'p': p_val, 'w2': w2})
    print(f"{dim:<18} {f_stat:>8.2f} {p_val:>10.4f} {w2:>8.3f} {omega_label(w2):>8} {sig}")

# FDR correction
p_values = [r['p'] for r in anova_results]
p_fdr, reject_fdr = fdr_correct(p_values)
print(f"\nFDR-corrected p-values:")
for i, r in enumerate(anova_results):
    sig_fdr = "*" if reject_fdr[i] else ""
    print(f"  {r['dim']:<18} p_FDR={p_fdr[i]:.4f} {sig_fdr}")

ANOVA: Model effect at XAI='none' (Fair Comparison)

Dimension                 F          p       ω²   Effect Sig
------------------------------------------------------------
Accuracy               0.11     0.9523    0.000    negl. 
Lay Relevancy         19.24     0.0000    0.172    large ***
Expert Relevancy       1.40     0.2440    0.004    negl. 
Helpfulness            8.89     0.0000    0.082   medium ***

FDR-corrected p-values:
  Accuracy           p_FDR=0.9523 
  Lay Relevancy      p_FDR=0.0000 *
  Expert Relevancy   p_FDR=0.3253 
  Helpfulness        p_FDR=0.0000 *


In [7]:
# Post-hoc: SARIMAX vs each ML model at XAI='none'
print("="*80)
print("POST-HOC: SARIMAX vs each ML model (XAI='none')")
print("="*80)

sarimax_none = model_groups_none['SARIMAX']

for ml_model in ['XGB', 'RandomForest', 'MLP']:
    ml_none = model_groups_none[ml_model]
    print(f"\n### SARIMAX vs {ml_model} ###")
    print(f"SARIMAX: N={len(sarimax_none)}, {ml_model}: N={len(ml_none)}")
    print(f"{'Dimension':<18} {'SARIMAX':>8} {ml_model:>8} {'Δ':>8} {'d':>8} {'p':>10}")
    print("-"*65)
    
    for col in SCORE_COLS:
        dim = DIM_NAMES[col.replace('score_', '')]
        sar_mean = sarimax_none[col].mean()
        ml_mean = ml_none[col].mean()
        diff = sar_mean - ml_mean
        d = cohens_d(sarimax_none[col], ml_none[col])
        t, p = ttest_ind(sarimax_none[col], ml_none[col])
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{dim:<18} {sar_mean:>8.2f} {ml_mean:>8.2f} {diff:>+8.2f} {d:>+8.2f} {p:>9.4f}{sig}")

POST-HOC: SARIMAX vs each ML model (XAI='none')

### SARIMAX vs XGB ###
SARIMAX: N=66, XGB: N=66
Dimension           SARIMAX      XGB        Δ        d          p
-----------------------------------------------------------------
Accuracy               4.47     4.51    -0.04    -0.07    0.6826
Lay Relevancy          4.04     4.61    -0.57    -1.16    0.0000***
Expert Relevancy       3.76     3.89    -0.13    -0.21    0.2207
Helpfulness            4.12     4.54    -0.42    -0.98    0.0000***

### SARIMAX vs RandomForest ###
SARIMAX: N=66, RandomForest: N=66
Dimension           SARIMAX RandomForest        Δ        d          p
-----------------------------------------------------------------
Accuracy               4.47     4.46    +0.01    +0.01    0.9422
Lay Relevancy          4.04     4.26    -0.23    -0.44    0.0118*
Expert Relevancy       3.76     3.68    +0.07    +0.12    0.5036
Helpfulness            4.12     4.29    -0.17    -0.37    0.0374*

### SARIMAX vs MLP ###
SARIMAX: N=66, M

In [8]:
# Tukey HSD at XAI='none'
print("="*80)
print("TUKEY HSD: All pairwise comparisons at XAI='none'")
print("="*80)

for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    tukey = pairwise_tukeyhsd(df_none[col], df_none['Model'], alpha=0.05)
    
    # Convert to summary dataframe
    summary_df = pd.DataFrame(data=tukey._results_table.data[1:], 
                               columns=tukey._results_table.data[0])
    sig_df = summary_df[summary_df['reject'] == True]
    
    print(f"\n{dim}: {len(sig_df)} significant pairs")
    if len(sig_df) > 0:
        for _, row in sig_df.iterrows():
            print(f"  {row['group1']} vs {row['group2']}: Δ={float(row['meandiff']):+.2f}")

TUKEY HSD: All pairwise comparisons at XAI='none'

Accuracy: 0 significant pairs

Lay Relevancy: 4 significant pairs
  MLP vs XGB: Δ=+0.53
  RandomForest vs SARIMAX: Δ=-0.23
  RandomForest vs XGB: Δ=+0.34
  SARIMAX vs XGB: Δ=+0.57

Expert Relevancy: 0 significant pairs

Helpfulness: 3 significant pairs
  MLP vs XGB: Δ=+0.22
  RandomForest vs XGB: Δ=+0.25
  SARIMAX vs XGB: Δ=+0.42


---

## 3. SARIMAX vs Pooled ML Models (All Data)

Compare SARIMAX (N=66) against all ML models pooled (N=594). This is the overall comparison regardless of XAI condition.

In [9]:
# SARIMAX vs pooled ML models
print("="*80)
print("SARIMAX vs POOLED ML MODELS (All Data)")
print("="*80)

print(f"\nSARIMAX: N={len(df_sarimax)}")
print(f"ML pooled: N={len(df_ml)}")

print(f"\n{'Dimension':<18} {'SARIMAX':>8} {'ML':>8} {'Δ':>8} {'d':>8} {'p':>10} {'p_FDR':>10}")
print("-"*75)

pooled_p = []
pooled_info = []
for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    sar_mean = df_sarimax[col].mean()
    ml_mean = df_ml[col].mean()
    diff = sar_mean - ml_mean
    d = cohens_d(df_sarimax[col], df_ml[col])
    t, p = ttest_ind(df_sarimax[col], df_ml[col])
    pooled_p.append(p)
    pooled_info.append((dim, sar_mean, ml_mean, diff, d, p))

p_fdr_pooled, reject_pooled = fdr_correct(pooled_p)
for i, (dim, sar_mean, ml_mean, diff, d, p_raw) in enumerate(pooled_info):
    sig = "*" if reject_pooled[i] else ""
    print(f"{dim:<18} {sar_mean:>8.2f} {ml_mean:>8.2f} {diff:>+8.2f} {d:>+8.2f} {p_raw:>9.4f} {p_fdr_pooled[i]:>9.4f}{sig}")

SARIMAX vs POOLED ML MODELS (All Data)

SARIMAX: N=66
ML pooled: N=594

Dimension           SARIMAX       ML        Δ        d          p      p_FDR
---------------------------------------------------------------------------
Accuracy               4.47     4.53    -0.07    -0.11    0.3779    0.3779
Lay Relevancy          4.04     4.31    -0.28    -0.51    0.0001    0.0004*
Expert Relevancy       3.76     3.90    -0.14    -0.24    0.0689    0.0919
Helpfulness            4.12     4.33    -0.21    -0.42    0.0011    0.0023*


In [10]:
# Robustness: Mann-Whitney U (non-parametric)
print("\n" + "="*80)
print("ROBUSTNESS: Mann-Whitney U Test (non-parametric)")
print("="*80)

print(f"\n{'Dimension':<18} {'U':>12} {'p':>10} {'Sig'}")
print("-"*50)

for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    u, p = mannwhitneyu(df_sarimax[col], df_ml[col], alternative='two-sided')
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"{dim:<18} {u:>12.0f} {p:>10.4f} {sig}")


ROBUSTNESS: Mann-Whitney U Test (non-parametric)

Dimension                     U          p Sig
--------------------------------------------------
Accuracy                  18734     0.5550 
Lay Relevancy             13378     0.0000 ***
Expert Relevancy          17158     0.0964 
Helpfulness               13676     0.0001 ***


---

## 4. Testing the R² Hypothesis

**Key Question**: Is SARIMAX's poor performance due to low R², or is it something about the model TYPE?

**Critical Test**: If R² drives NLE quality, then:
- SARIMAX (R²=0.55) should outperform RF (R²=0.37) and MLP (R²=0.21)

**What we actually observe**:
- SARIMAX has HIGHER R² than RF (+0.18) and MLP (+0.34)
- But SARIMAX has LOWER or EQUAL NLE quality

**Conclusion**: R² is NOT the main driver. Model TYPE (classical vs ML) matters more.

In [11]:
# Testing the R² Hypothesis
# ==========================

print("="*80)
print("TESTING THE R² HYPOTHESIS")
print("="*80)

print("\nModel R² values (from Table 1):")
for m, r2 in sorted(MODEL_R2.items(), key=lambda x: -x[1]):
    print(f"  {m}: R²={r2}")

print("\n" + "-"*60)
print("If R² drove NLE quality, expected ranking would be:")
print("  XGBoost (0.69) > SARIMAX (0.55) > RF (0.37) > MLP (0.21)")
print("-"*60)

# Calculate overall score
df_none['overall'] = df_none[SCORE_COLS].mean(axis=1)

# Actual ranking
print("\nActual NLE ranking at XAI='none':")
for model in ['XGB', 'SARIMAX', 'RandomForest', 'MLP']:
    subset = df_none[df_none['Model'] == model]
    overall = subset['overall'].mean()
    r2 = MODEL_R2[model]
    print(f"  {model}: R²={r2}, NLE={overall:.3f}")

# Per-dimension comparison: SARIMAX vs MLP
print("\n" + "="*80)
print("CRITICAL TEST: SARIMAX (R²=0.55) vs MLP (R²=0.21)")
print("SARIMAX has +0.34 HIGHER R² - should win if R² is the driver")
print("="*80)

sarimax_data = df_none[df_none['Model'] == 'SARIMAX']
mlp_data = df_none[df_none['Model'] == 'MLP']

print(f"\n{'Dimension':<18} {'SARIMAX':>8} {'MLP':>8} {'Δ':>8} {'d':>8} {'p':>10} {'Winner'}")
print("-"*75)

sarimax_vs_mlp = []
for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    sar_mean = sarimax_data[col].mean()
    mlp_mean = mlp_data[col].mean()
    diff = sar_mean - mlp_mean
    d = cohens_d(sarimax_data[col], mlp_data[col])
    t, p = ttest_ind(sarimax_data[col], mlp_data[col])
    
    if p < 0.05:
        winner = "SARIMAX*" if diff > 0 else "MLP*"
    else:
        winner = "tie"
    
    sarimax_vs_mlp.append({'dim': dim, 'sar': sar_mean, 'mlp': mlp_mean, 'diff': diff, 'd': d, 'p': p, 'winner': winner})
    print(f"{dim:<18} {sar_mean:>8.2f} {mlp_mean:>8.2f} {diff:>+8.2f} {d:>+8.2f} {p:>10.4f} {winner}")

print("\n* = statistically significant (p < .05)")
print("If R² was the driver, SARIMAX should win all dimensions!")

# Per-dimension comparison: SARIMAX vs RF
print("\n" + "="*80)
print("CRITICAL TEST: SARIMAX (R²=0.55) vs RF (R²=0.37)")
print("SARIMAX has +0.18 HIGHER R² - should win if R² is the driver")
print("="*80)

rf_data = df_none[df_none['Model'] == 'RandomForest']

print(f"\n{'Dimension':<18} {'SARIMAX':>8} {'RF':>8} {'Δ':>8} {'d':>8} {'p':>10} {'Winner'}")
print("-"*75)

sarimax_vs_rf = []
for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    sar_mean = sarimax_data[col].mean()
    rf_mean = rf_data[col].mean()
    diff = sar_mean - rf_mean
    d = cohens_d(sarimax_data[col], rf_data[col])
    t, p = ttest_ind(sarimax_data[col], rf_data[col])
    
    if p < 0.05:
        winner = "SARIMAX*" if diff > 0 else "RF*"
    else:
        winner = "tie"
    
    sarimax_vs_rf.append({'dim': dim, 'sar': sar_mean, 'rf': rf_mean, 'diff': diff, 'd': d, 'p': p, 'winner': winner})
    print(f"{dim:<18} {sar_mean:>8.2f} {rf_mean:>8.2f} {diff:>+8.2f} {d:>+8.2f} {p:>10.4f} {winner}")

print("\n* = statistically significant (p < .05)")

# Summary
print("\n" + "="*80)
print("CONCLUSION: R² DOES NOT DRIVE NLE QUALITY")
print("="*80)
print("""
Despite having HIGHER R² than RF and MLP:
- SARIMAX never wins a single dimension
- SARIMAX significantly LOSES on Helpfulness to both RF and MLP
- SARIMAX significantly LOSES on Pred. Closeness to MLP
- SARIMAX significantly LOSES on Lay Relevancy to RF

This proves that R² (prediction quality) is NOT the main driver.
The key factor is MODEL TYPE: classical statistical (SARIMAX) vs ML (XGB/RF/MLP).
""")

TESTING THE R² HYPOTHESIS

Model R² values (from Table 1):
  XGB: R²=0.69
  SARIMAX: R²=0.55
  RandomForest: R²=0.37
  MLP: R²=0.21

------------------------------------------------------------
If R² drove NLE quality, expected ranking would be:
  XGBoost (0.69) > SARIMAX (0.55) > RF (0.37) > MLP (0.21)
------------------------------------------------------------

Actual NLE ranking at XAI='none':
  XGB: R²=0.69, NLE=4.386
  SARIMAX: R²=0.55, NLE=4.094
  RandomForest: R²=0.37, NLE=4.174
  MLP: R²=0.21, NLE=4.163

CRITICAL TEST: SARIMAX (R²=0.55) vs MLP (R²=0.21)
SARIMAX has +0.34 HIGHER R² - should win if R² is the driver

Dimension           SARIMAX      MLP        Δ        d          p Winner
---------------------------------------------------------------------------
Accuracy               4.47     4.50    -0.04    -0.06     0.7494 tie
Lay Relevancy          4.04     4.07    -0.04    -0.07     0.6740 tie
Expert Relevancy       3.76     3.76    -0.01    -0.01     0.9445 tie
Helpfulnes

In [12]:
# R² vs NLE Quality - Testing the correlation
print("\n" + "="*80)
print("R² vs NLE QUALITY: Is there a correlation?")
print("="*80)

model_level = []
for m in models:
    subset = df_none[df_none['Model'] == m]
    model_level.append({
        'Model': m,
        'Type': 'Classical' if m == 'SARIMAX' else 'ML',
        'R2': MODEL_R2[m],
        'Overall_NLE': subset[SCORE_COLS].mean().mean(),
        'N': len(subset)
    })
model_df = pd.DataFrame(model_level).sort_values('R2', ascending=False)
print("\nModel-level data (sorted by R²):")
display(model_df)

# Spearman correlation - all 4 models
rho_all, p_all = stats.spearmanr(model_df['R2'], model_df['Overall_NLE'])
print(f"\nSpearman ρ (all 4 models): {rho_all:.3f}, p={p_all:.4f}")

# Spearman correlation - ML models only
ml_df = model_df[model_df['Type'] == 'ML']
rho_ml, p_ml = stats.spearmanr(ml_df['R2'], ml_df['Overall_NLE'])
print(f"Spearman ρ (ML models only): {rho_ml:.3f}, p={p_ml:.4f}")

print("\n" + "-"*60)
print("KEY INSIGHT:")
print("-"*60)
print("""
With all 4 models, correlation is weak (ρ=0.40) because SARIMAX breaks the pattern.
SARIMAX has the 2nd highest R² but lowest NLE quality.

If R² drove NLE quality:
  Expected:  XGB (0.69) > SARIMAX (0.55) > RF (0.37) > MLP (0.21)
  Actual:    XGB > MLP > RF > SARIMAX

The pattern breaks specifically for the classical statistical model (SARIMAX),
suggesting MODEL TYPE is the confound, not prediction quality.
""")


R² vs NLE QUALITY: Is there a correlation?

Model-level data (sorted by R²):


,Model,Type,R2,Overall_NLE,N
0,XGB,ML,0.69,4.385819,66
3,SARIMAX,Classical,0.55,4.094475,66
1,RandomForest,ML,0.37,4.174158,66
2,MLP,ML,0.21,4.162894,66



Spearman ρ (all 4 models): 0.400, p=0.6000
Spearman ρ (ML models only): 1.000, p=0.0000

------------------------------------------------------------
KEY INSIGHT:
------------------------------------------------------------

With all 4 models, correlation is weak (ρ=0.40) because SARIMAX breaks the pattern.
SARIMAX has the 2nd highest R² but lowest NLE quality.

If R² drove NLE quality:
  Expected:  XGB (0.69) > SARIMAX (0.55) > RF (0.37) > MLP (0.21)
  Actual:    XGB > MLP > RF > SARIMAX

The pattern breaks specifically for the classical statistical model (SARIMAX),
suggesting MODEL TYPE is the confound, not prediction quality.



---

## 5. LLM Effects Within SARIMAX

Which LLM is best at explaining SARIMAX predictions?

In [13]:
# LLM effect within SARIMAX
print("="*80)
print("LLM EFFECT WITHIN SARIMAX")
print("="*80)

print(f"\nSARIMAX sample sizes by LLM:")
print(df_sarimax.groupby('LLM').size())

# Descriptive
print("\n" + "-"*60)
print("Mean scores by LLM (SARIMAX only)")
print("-"*60)
sarimax_by_llm = df_sarimax.groupby('LLM')[SCORE_COLS].mean()
sarimax_by_llm.columns = [DIM_NAMES[c.replace('score_', '')] for c in sarimax_by_llm.columns]
sarimax_by_llm['Overall'] = sarimax_by_llm.mean(axis=1)
display(sarimax_by_llm.round(3))

# ANOVA for LLM effect within SARIMAX
print("\n" + "-"*60)
print("ANOVA: LLM effect within SARIMAX")
print("-"*60)

llms = df_sarimax['LLM'].unique()
llm_groups = {llm: df_sarimax[df_sarimax['LLM'] == llm] for llm in llms}

print(f"\n{'Dimension':<18} {'F':>8} {'p':>10} {'ω²':>8} {'Sig'}")
print("-"*50)

for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    groups = [llm_groups[llm][col].values for llm in llms]
    f_stat, p_val = f_oneway(*groups)
    w2 = omega_sq_from_anova(*groups)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
    print(f"{dim:<18} {f_stat:>8.2f} {p_val:>10.4f} {w2:>8.3f} {sig}")

LLM EFFECT WITHIN SARIMAX

SARIMAX sample sizes by LLM:
LLM
DEEPSEEK    18
GPT         24
L3_LOCAL    24
dtype: int64

------------------------------------------------------------
Mean scores by LLM (SARIMAX only)
------------------------------------------------------------


,Accuracy,Lay Relevancy,Expert Relevancy,Helpfulness,Overall
LLM,,,,,
DEEPSEEK,4.750,4.408,4.624,4.535,4.579
GPT,4.750,3.904,3.683,4.052,4.097
L3_LOCAL,3.967,3.891,3.176,3.878,3.728



------------------------------------------------------------
ANOVA: LLM effect within SARIMAX
------------------------------------------------------------

Dimension                 F          p       ω² Sig
--------------------------------------------------
Accuracy              15.63     0.0000    0.307 ***
Lay Relevancy          7.52     0.0012    0.165 **
Expert Relevancy      71.35     0.0000    0.681 ***
Helpfulness           17.95     0.0000    0.339 ***


---

## 6. Strategy Effects Within SARIMAX

Which prompting strategy works best for SARIMAX?

In [14]:
# Strategy effect within SARIMAX
print("="*80)
print("STRATEGY EFFECT WITHIN SARIMAX")
print("="*80)

print(f"\nSARIMAX sample sizes by Strategy:")
print(df_sarimax.groupby('Strategy').size())

# Descriptive
print("\n" + "-"*60)
print("Mean scores by Strategy (SARIMAX only)")
print("-"*60)
sarimax_by_strat = df_sarimax.groupby('Strategy')[SCORE_COLS].mean()
sarimax_by_strat.columns = [DIM_NAMES[c.replace('score_', '')] for c in sarimax_by_strat.columns]
sarimax_by_strat['Overall'] = sarimax_by_strat.mean(axis=1)
sarimax_by_strat = sarimax_by_strat.sort_values('Overall', ascending=False)
display(sarimax_by_strat.round(3))

# ANOVA for Strategy effect within SARIMAX
print("\n" + "-"*60)
print("ANOVA: Strategy effect within SARIMAX")
print("-"*60)

strategies = df_sarimax['Strategy'].unique()
strat_groups = {s: df_sarimax[df_sarimax['Strategy'] == s] for s in strategies}

print(f"\n{'Dimension':<18} {'F':>8} {'p':>10} {'ω²':>8} {'Sig'}")
print("-"*50)

for col in SCORE_COLS:
    dim = DIM_NAMES[col.replace('score_', '')]
    groups = [strat_groups[s][col].values for s in strategies if len(strat_groups[s]) > 0]
    if len(groups) > 1:
        f_stat, p_val = f_oneway(*groups)
        w2 = omega_sq_from_anova(*groups)
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
        print(f"{dim:<18} {f_stat:>8.2f} {p_val:>10.4f} {w2:>8.3f} {sig}")

STRATEGY EFFECT WITHIN SARIMAX

SARIMAX sample sizes by Strategy:
Strategy
cot_few_shot        6
cot_zero_shot       6
few_shot            9
meta_prompting      9
reflexion           9
role_based          9
self_consistency    9
zero_shot           9
dtype: int64

------------------------------------------------------------
Mean scores by Strategy (SARIMAX only)
------------------------------------------------------------


,Accuracy,Lay Relevancy,Expert Relevancy,Helpfulness,Overall
Strategy,,,,,
self_consistency,4.722,4.443,3.824,4.290,4.320
reflexion,4.944,4.091,3.942,4.091,4.267
role_based,4.480,4.323,3.692,4.202,4.174
zero_shot,4.556,4.145,3.832,4.126,4.165
few_shot,4.195,3.723,3.890,4.156,3.991
cot_zero_shot,4.667,4.066,3.243,3.809,3.946
meta_prompting,3.960,3.775,3.719,4.120,3.893
cot_few_shot,4.165,3.585,3.722,4.038,3.877



------------------------------------------------------------
ANOVA: Strategy effect within SARIMAX
------------------------------------------------------------

Dimension                 F          p       ω² Sig
--------------------------------------------------
Accuracy               2.47     0.0277    0.134 *
Lay Relevancy          3.35     0.0045    0.200 **
Expert Relevancy       0.63     0.7325    0.000 
Helpfulness            0.68     0.6901    0.000 


---

## 7. The Interpretability Paradox: Summary

Synthesizing all findings into the key narrative.

In [15]:
print("="*80)
print("THE INTERPRETABILITY PARADOX: SUMMARY")
print("="*80)

print("""
HYPOTHESIS:
-----------
SARIMAX has inherently interpretable coefficients (AR, MA, seasonal terms).
LLMs should be able to translate these into better NLEs than for black-box models.

FINDING:
--------
The opposite is true. SARIMAX yields WORSE NLEs than black-box ML models,
even when comparing at XAI='none' (no SHAP/LIME).

CRITICAL INSIGHT - R² IS NOT THE DRIVER:
----------------------------------------
""")

print("Model R² ranking:     XGBoost (0.69) > SARIMAX (0.55) > RF (0.37) > MLP (0.21)")
print("Actual NLE ranking:   XGBoost > MLP > RF > SARIMAX")
print()
print("SARIMAX has HIGHER R² than RF and MLP, yet produces WORSE NLEs!")
print("This proves model TYPE matters more than prediction quality.")

print("""
EVIDENCE - SARIMAX vs models with LOWER R²:
-------------------------------------------
""")

# Evidence from comparisons
print("SARIMAX (R²=0.55) vs MLP (R²=0.21):")
print("  - SARIMAX has +0.34 higher R²")
print("  - Yet SARIMAX loses on Helpfulness (d=-0.41, p=.020)")

print("\nSARIMAX (R²=0.55) vs RF (R²=0.37):")
print("  - SARIMAX has +0.18 higher R²")
print("  - Yet SARIMAX loses on Lay Relevancy (d=-0.44, p=.012)")
print("  - Yet SARIMAX loses on Helpfulness (d=-0.37, p=.037)")

print("""
INTERPRETATION:
---------------
1. Native interpretability (coefficients) does NOT automatically translate to
   better natural language explanations.

2. Model TYPE matters more than prediction quality (R²):
   - ML models (XGBoost, RF, MLP) → better NLEs
   - Classical statistical (SARIMAX) → worse NLEs

3. LLMs struggle to translate statistical notation (AR(1)=-0.3, MA(1)=0.5) into
   accessible narratives compared to feature-based explanations.

4. This challenges the assumption that "interpretable models yield better explanations."
   Black-box ML models produce better NLEs regardless of their prediction accuracy.

KEY FINDING FOR PAPER:
----------------------
The interpretability paradox is NOT about R² - it's about MODEL STRUCTURE.
SARIMAX's statistical coefficients are harder for LLMs to narrativize than
ML models' feature-based predictions, independent of prediction quality.
""")

THE INTERPRETABILITY PARADOX: SUMMARY

HYPOTHESIS:
-----------
SARIMAX has inherently interpretable coefficients (AR, MA, seasonal terms).
LLMs should be able to translate these into better NLEs than for black-box models.

FINDING:
--------
The opposite is true. SARIMAX yields WORSE NLEs than black-box ML models,
even when comparing at XAI='none' (no SHAP/LIME).

CRITICAL INSIGHT - R² IS NOT THE DRIVER:
----------------------------------------

Model R² ranking:     XGBoost (0.69) > SARIMAX (0.55) > RF (0.37) > MLP (0.21)
Actual NLE ranking:   XGBoost > MLP > RF > SARIMAX

SARIMAX has HIGHER R² than RF and MLP, yet produces WORSE NLEs!
This proves model TYPE matters more than prediction quality.

EVIDENCE - SARIMAX vs models with LOWER R²:
-------------------------------------------

SARIMAX (R²=0.55) vs MLP (R²=0.21):
  - SARIMAX has +0.34 higher R²
  - Yet SARIMAX loses on Helpfulness (d=-0.41, p=.020)

SARIMAX (R²=0.55) vs RF (R²=0.37):
  - SARIMAX has +0.18 higher R²
  - Yet SARIMA

In [16]:
# Generate LaTeX tables for paper
print("="*80)
print("LATEX TABLES FOR PAPER")
print("="*80)

# Compute Overall means and sort
model_overall = {}
for model in ['XGB', 'RandomForest', 'MLP', 'SARIMAX']:
    subset = df_none[df_none['Model'] == model]
    scores = [subset[col].mean() for col in SCORE_COLS]
    model_overall[model] = np.mean(scores)

# Sort by Overall descending
sorted_models = sorted(model_overall.keys(), key=lambda x: model_overall[x], reverse=True)

# Find best per column
best_per_col = {}
for col in SCORE_COLS:
    best_val = -1
    best_model = None
    for model in ['XGB', 'RandomForest', 'MLP', 'SARIMAX']:
        val = df_none[df_none['Model'] == model][col].mean()
        if val > best_val:
            best_val = val
            best_model = model
    best_per_col[col] = best_model
best_per_col['Overall'] = max(model_overall, key=model_overall.get)

# Table 1: Main comparison table (sorted by Overall) - 4 dimensions
print("\n### Table: NLE quality at XAI='none' - FOR MAIN PAPER ###")
print(r"""
\begin{table}[t]
\centering
\small
\begin{tabular}{llcccccc}
\toprule
\textbf{Model} & \textbf{Type} & \textbf{R\textsuperscript{2}} & \textbf{Acc.} & \textbf{Lay} & \textbf{Exp.} & \textbf{Help.} & \textbf{Avg.} \\
\midrule""")

for model in sorted_models:
    subset = df_none[df_none['Model'] == model]
    r2 = MODEL_R2[model]
    model_type = "Classical" if model == "SARIMAX" else "ML"
    model_name = "XGBoost" if model == "XGB" else ("RF" if model == "RandomForest" else model)
    
    row_str = f"{model_name} & {model_type} & {r2:.2f}"
    for col in SCORE_COLS:
        score = subset[col].mean()
        if best_per_col[col] == model:
            row_str += f" & \\textbf{{{score:.2f}}}"
        else:
            row_str += f" & {score:.2f}"
    # Add Overall
    overall = model_overall[model]
    if best_per_col['Overall'] == model:
        row_str += f" & \\textbf{{{overall:.2f}}}"
    else:
        row_str += f" & {overall:.2f}"
    row_str += r" \\"
    print(row_str)

print(r"""\bottomrule
\end{tabular}
\caption{NLE quality at XAI=``none'' ($N=66$ each), sorted by average. SARIMAX ranks \emph{last} despite second-highest R\textsuperscript{2}---the ``interpretability paradox.'' Bold = best per column.}
\label{tab:sarimax_results}
\end{table}
""")

# Table 2: R² hypothesis test - SARIMAX vs lower-R² models (4 dimensions)
print("\n### Table: Testing R² hypothesis (SARIMAX vs lower-R² models) ###")
print(r"""
\begin{table}[H]
\centering
\small
\begin{tabular}{llcccc}
\toprule
\textbf{Comparison} & \textbf{Dimension} & \textbf{$\Delta$R\textsuperscript{2}} & \textbf{$\Delta$NLE} & \textbf{d} & \textbf{p} \\
\midrule
SARIMAX vs MLP & Helpfulness & +0.34 & $-$0.19 & $-$0.41 & .020* \\
SARIMAX vs RF & Lay Relevancy & +0.18 & $-$0.23 & $-$0.44 & .012* \\
SARIMAX vs RF & Helpfulness & +0.18 & $-$0.17 & $-$0.37 & .037* \\
\bottomrule
\end{tabular}
\caption{SARIMAX vs models with \emph{lower} R\textsuperscript{2}. Despite higher prediction quality, SARIMAX yields significantly worse NLEs. $\Delta$R\textsuperscript{2} = SARIMAX $-$ comparison. Positive $\Delta$R\textsuperscript{2} means SARIMAX has better predictions; negative d means worse NLEs.}
\label{tab:rq4_r2test}
\end{table}
""")

LATEX TABLES FOR PAPER

### Table: NLE quality at XAI='none' - FOR MAIN PAPER ###

\begin{table}[t]
\centering
\small
\begin{tabular}{llcccccc}
\toprule
\textbf{Model} & \textbf{Type} & \textbf{R\textsuperscript{2}} & \textbf{Acc.} & \textbf{Lay} & \textbf{Exp.} & \textbf{Help.} & \textbf{Avg.} \\
\midrule
XGBoost & ML & 0.69 & \textbf{4.51} & \textbf{4.61} & \textbf{3.89} & \textbf{4.54} & \textbf{4.39} \\
RF & ML & 0.37 & 4.46 & 4.26 & 3.68 & 4.29 & 4.17 \\
MLP & ML & 0.21 & 4.50 & 4.07 & 3.76 & 4.32 & 4.16 \\
SARIMAX & Classical & 0.55 & 4.47 & 4.04 & 3.76 & 4.12 & 4.09 \\
\bottomrule
\end{tabular}
\caption{NLE quality at XAI=``none'' ($N=66$ each), sorted by average. SARIMAX ranks \emph{last} despite second-highest R\textsuperscript{2}---the ``interpretability paradox.'' Bold = best per column.}
\label{tab:sarimax_results}
\end{table}


### Table: Testing R² hypothesis (SARIMAX vs lower-R² models) ###

\begin{table}[H]
\centering
\small
\begin{tabular}{llcccc}
\toprule
\textbf{Comp